# Case 1 · Do win-back discounts work? (Part 2)
**BUS-G 492 · Causal decisions with AI**

In [1]:
# --- Setup: run this cell first -----------------------------------------------
import os
DATA_URL = "https://raw.githubusercontent.com/boyoung-seo/492-coding/main/cases/case1_rct_retention/data/"
for candidate in ["data/", "cases/case1_rct_retention/data/", "../../../cases/case1_rct_retention/data/", "../../../../cases/case1_rct_retention/data/"]:
    if os.path.exists(candidate + "customers_history.csv"):
        DATA = candidate
        break
else:
    DATA = DATA_URL
print("Reading data from:", DATA)

Reading data from: https://raw.githubusercontent.com/boyoung-seo/492-coding/main/cases/case1_rct_retention/data/


## Stage 5 · Something you didn't know

> **From:** Luis Ortega, Retention Team Lead
> **Re:** how agents decide who gets the offer
>
> We don't pick at random. In the CRM we can see who visited the **cancel page** in the last week and what they said in **chat**. If someone is clearly on their way out, we send the offer. None of that makes it into the analytics warehouse; the risk score doesn't see it either.
>
> Also, you may not know this: in **April 2026 we ran a test**. Among 8,000 customers with risk_score ≥ 30, a **coin flip** decided who got the offer. The results are in `april_discount_test.csv`.

In [5]:
import pandas as pd
test = pd.read_csv(DATA + "april_discount_test.csv")
print(test.shape)
test.head()

(8000, 13)


,customer_id,region,plan,tenure_months,orders_last_90d,avg_order_value,skipped_weeks_90d,support_tickets_90d,app_sessions_per_week,risk_score,assigned_discount,churned_90d,revenue_next_90d
0,100000,Midwest,2-person,29,8,68.41,3,0,2.7,37,0,0,506.58
1,100001,Northeast,family,27,3,85.68,0,0,0.7,39,1,1,16.61
2,100002,Midwest,family,2,6,90.24,0,3,3.9,63,1,0,402.62
3,100003,South,family,13,9,103.06,2,1,2.4,55,1,1,80.70
4,100004,West,2-person,16,7,59.94,2,0,0.3,52,0,1,7.40


### Your tasks

Use AI again, but this time **tell it the whole story**: how the data were generated, and what the new information is.

1. **Check the coin flip.** Compare the two groups on everything measured *before* the offer (tenure, orders, risk score, etc.). Should they differ?

| Measured before the offer | No offer | Offer | T-stat |
| --- | --- | --- | --- |
|tenure_months | | | |
|orders_last_90d | | | |
|skipped_weeks_90d | | | |
|suppoert_tickets_90d | | | |
|app sessions_per_week | | | |
|risk_score | | | |

2. **Effect on the treated (ETT).** Estimate the effect of the offer on churn **for the customers who received it**, with a 95% confidence interval. In this test every customer assigned the offer received it, so the ETT equals the simple difference in churn by assignment (the intention-to-treat effect, ITT). Why is a simple difference in means enough here? How would you get the ETT if only some of the assigned customers had actually received the offer?


In [7]:
# 1. Balance check
print(test.columns)
print(test.dtypes)

import pandas as pd
from scipy.stats import ttest_ind

# Variables measured before the offer
balance_vars = [
    "tenure_months",
    "orders_last_90d",
    "skipped_weeks_90d",
    "support_tickets_90d",
    "app_sessions_per_week",
    "risk_score"
]

rows = []

for var in balance_vars:
    no_offer = test.loc[test["assigned_discount"] == 0, var].dropna()
    offer = test.loc[test["assigned_discount"] == 1, var].dropna()

    # Welch two-sample t-test: Offer - No offer
    t_stat, p_value = ttest_ind(
        offer,
        no_offer,
        equal_var=False
    )

    rows.append({
        "Measured before the offer": var,
        "No offer": no_offer.mean(),
        "Offer": offer.mean(),
        "T-stat": t_stat
    })

balance_table = pd.DataFrame(rows)

# Round for display
balance_table["No offer"] = balance_table["No offer"].round(2)
balance_table["Offer"] = balance_table["Offer"].round(2)
balance_table["T-stat"] = balance_table["T-stat"].round(2)

print(balance_table.to_string(index=False))

Index(['customer_id', 'region', 'plan', 'tenure_months', 'orders_last_90d',
       'avg_order_value', 'skipped_weeks_90d', 'support_tickets_90d',
       'app_sessions_per_week', 'risk_score', 'assigned_discount',
       'churned_90d', 'revenue_next_90d'],
      dtype='object')
customer_id                int64
region                    object
plan                      object
tenure_months              int64
orders_last_90d            int64
avg_order_value          float64
skipped_weeks_90d          int64
support_tickets_90d        int64
app_sessions_per_week    float64
risk_score                 int64
assigned_discount          int64
churned_90d                int64
revenue_next_90d         float64
dtype: object
Measured before the offer  No offer  Offer  T-stat
            tenure_months     13.00  12.99   -0.03
          orders_last_90d      6.47   6.44   -0.47
        skipped_weeks_90d      2.49   2.47   -0.48
      support_tickets_90d      0.69   0.68   -0.17
    app_sessions_per_wee

In [8]:
# 2. Effect on the treated (ETT)
# Treatment and control groups
treated = test[test["assigned_discount"] == 1]
control = test[test["assigned_discount"] == 0]

# Churn rates
churn_treated = treated["churned_90d"].mean()
churn_control = control["churned_90d"].mean()

# Treatment effect on the treated
att = churn_treated - churn_control

print(f"Churn rate - offer:    {churn_treated:.2%}")
print(f"Churn rate - no offer: {churn_control:.2%}")
print(f"ATT: {att:.2%}")
print(f"ATT: {att * 100:.2f} percentage points")

Churn rate - offer:    29.95%
Churn rate - no offer: 41.22%
ATT: -11.26%
ATT: -11.26 percentage points


In [9]:
import statsmodels.api as sm

X = sm.add_constant(test["assigned_discount"])

model = sm.OLS(test["churned_90d"], X).fit(cov_type="HC1")

print(model.summary())

att = model.params["assigned_discount"]
ci = model.conf_int().loc["assigned_discount"]

print(f"\nEstimated effect: {att * 100:.2f} percentage points")
print(
    f"95% CI: [{ci.iloc[0] * 100:.2f}, "
    f"{ci.iloc[1] * 100:.2f}] percentage points"
)

                            OLS Regression Results                            
Dep. Variable:            churned_90d   R-squared:                       0.014
Model:                            OLS   Adj. R-squared:                  0.014
Method:                 Least Squares   F-statistic:                     112.3
Date:                Thu, 01 Oct 2026   Prob (F-statistic):           4.63e-26
Time:                        15:36:18   Log-Likelihood:                -5408.1
No. Observations:                8000   AIC:                         1.082e+04
Df Residuals:                    7998   BIC:                         1.083e+04
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                 0.4122      0.00

### Decide again

In [ ]:
TEAM = ""                  # e.g. "Team 3: Ana & Ben"  (use the same name every time)
RISK_THRESHOLD = None      # offer goes to customers with risk_score >= this; 101 = nobody  (e.g. 60)
PREDICTED_PROFIT = None    # in dollars, a plain number: 150000, not "$150,000"
JUSTIFICATION = """
Three sentences for the decision maker: what you found, what you recommend, and why.
"""

# Stops with a message if anything is still blank, so you don't submit an empty field.
blank = [name for name, value in [("TEAM", TEAM), ("RISK_THRESHOLD", RISK_THRESHOLD), ("PREDICTED_PROFIT", PREDICTED_PROFIT)] if value in (None, "")]
if blank:
    raise ValueError("Fill these in before submitting: " + ", ".join(blank))
PREDICTED_PROFIT = float(PREDICTED_PROFIT)   # must be a number
print("Enter exactly these values in the submission form:")
print()
print(f"team:              {TEAM}")
print("case:              1")
print("round:             2  (Part II)")
print(f"decision:          {RISK_THRESHOLD}")
print(f"predicted_profit:  {PREDICTED_PROFIT:.0f}")

In [20]:
import numpy as np
import pandas as pd
df = pd.read_csv(DATA + "customers_history.csv")

# ---------------------------------------------------------
# Settings
# ---------------------------------------------------------

ATT = -0.1126
CHURN_REDUCTION = -ATT

N_FUTURE = 30000
N_SIMULATIONS = 1000
SEED = 42

rng = np.random.default_rng(SEED)

# Change this if your observational DataFrame has another name
history = df.copy()

# Candidate thresholds
thresholds = np.arange(
    history["risk_score"].min(),
    102
)

# T = 101 means no offers
thresholds = np.unique(np.append(thresholds, 101))


# ---------------------------------------------------------
# Run simulations
# ---------------------------------------------------------

simulation_results = []

for sim in range(N_SIMULATIONS):

    # Sample 30,000 future customers from historical population
    future = history.sample(
        n=N_FUTURE,
        replace=True,
        random_state=SEED + sim
    ).copy()

    risk = future["risk_score"].to_numpy()

    # Simulated churn WITHOUT an offer
    churn_no_offer = future["churned_90d"].to_numpy().copy()

    for T in thresholds:

        offered = risk >= T

        n_offered = offered.sum()

        if n_offered == 0:
            incremental_profit = 0.0

        else:
            # ---------------------------------------------
            # Simulate churn under offer
            # ---------------------------------------------
            churn_with_offer = churn_no_offer.copy()

            # ATT = -11.26 percentage points.
            #
            # Among customers who would otherwise churn,
            # randomly "save" enough customers to generate
            # the experimental average treatment effect.

            eligible = np.where(
                offered & (churn_no_offer == 1)
            )[0]

            # Each targeted customer has an expected
            # 0.1126 reduction in churn.
            #
            # Number saved is drawn probabilistically.
            n_saved_expected = CHURN_REDUCTION * n_offered

            n_saved = rng.binomial(
                n=n_offered,
                p=CHURN_REDUCTION
            )

            # Cannot save more churners than exist
            n_saved = min(n_saved, len(eligible))

            if n_saved > 0:
                saved = rng.choice(
                    eligible,
                    size=n_saved,
                    replace=False
                )

                churn_with_offer[saved] = 0

            # ---------------------------------------------
            # Incremental profit for targeted customers
            # ---------------------------------------------
            #
            # (churn_without - churn_with) * 800
            # - (1 - churn_with) * 90
            # - churn_with * 20

            individual_profit = (
                (churn_no_offer - churn_with_offer) * 800
                - (1 - churn_with_offer) * 90
                - churn_with_offer * 20
            )

            # No offer -> no incremental profit
            individual_profit[~offered] = 0

            incremental_profit = individual_profit.sum()

        simulation_results.append({
            "simulation": sim,
            "T": T,
            "n_offered": n_offered,
            "offer_rate": n_offered / N_FUTURE,
            "incremental_profit": incremental_profit
        })


simulation_results = pd.DataFrame(simulation_results)


# ---------------------------------------------------------
# Average across the 1,000 simulations
# ---------------------------------------------------------

summary = (
    simulation_results
    .groupby("T")
    .agg(
        expected_offers=("n_offered", "mean"),
        offer_rate=("offer_rate", "mean"),
        expected_incremental_profit=(
            "incremental_profit", "mean"
        ),
        sd_profit=("incremental_profit", "std")
    )
    .reset_index()
)

# Find optimal T
best = summary.loc[
    summary["expected_incremental_profit"].idxmax()
]


# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

print("OPTIMAL THRESHOLD FOR NEXT 30,000")
print("-----------------------------------")
print(f"Optimal T: {best['T']:.0f}")
print(f"Expected offers: {best['expected_offers']:,.0f}")
print(f"Offer rate: {best['offer_rate']:.1%}")

print(
    f"Expected incremental profit vs T=101: "
    f"${best['expected_incremental_profit']:,.2f}"
)

print(
    f"Simulation SD of profit: "
    f"${best['sd_profit']:,.2f}"
)


print("\nTOP 10 THRESHOLDS")
print(
    summary
    .sort_values(
        "expected_incremental_profit",
        ascending=False
    )
    .head(10)
    .to_string(index=False)
)

OPTIMAL THRESHOLD FOR NEXT 30,000
-----------------------------------
Optimal T: 1
Expected offers: 30,000
Offer rate: 100.0%
Expected incremental profit vs T=101: $462,297.00
Simulation SD of profit: $40,023.79

TOP 10 THRESHOLDS
 T  expected_offers  offer_rate  expected_incremental_profit    sd_profit
 1        30000.000    1.000000                    462297.00 40023.785871
 3        29248.724    0.974957                    461299.79 39903.038151
 5        28649.488    0.954983                    460595.98 39842.399137
 2        29550.519    0.985017                    460254.14 40551.523233
 4        28950.310    0.965010                    459537.22 37976.561101
 6        28352.441    0.945081                    457873.91 39787.184037
 7        28051.695    0.935057                    456150.64 39866.983685
 8        27752.593    0.925086                    455755.98 38391.770393
 9        27451.759    0.915059                    454259.28 39323.281587
10        27154.193    0.9051

In [14]:
import numpy as np
import pandas as pd

N_FUTURE = 30000

# Candidate thresholds, including T=101 = no offers
thresholds = sorted(test["risk_score"].unique())
thresholds.append(101)

results = []

for T in thresholds:

    # Customers who would be offered under this threshold
    target = test[test["risk_score"] >= T]

    # T = 101: no offers, so incremental profit = 0
    if len(target) == 0:
        results.append({
            "T": T,
            "target_rate": 0,
            "churn_no_offer": np.nan,
            "churn_offer": np.nan,
            "drop_in_churn": np.nan,
            "value_per_targeted": 0,
            "expected_offers_30000": 0,
            "profit_30000": 0
        })
        continue

    no_offer = target[target["assigned_discount"] == 0]
    offer = target[target["assigned_discount"] == 1]

    # Require both experimental groups
    if len(no_offer) == 0 or len(offer) == 0:
        continue

    # Experimental churn rates
    p0 = no_offer["churned_90d"].mean()
    p1 = offer["churned_90d"].mean()

    # Causal reduction in churn
    churn_drop = p0 - p1

    # Incremental value per targeted customer
    value_per_targeted = (
        churn_drop * 800
        - (1 - p1) * 90
        - p1 * 20
    )

    # Fraction of future customers who would receive offer
    target_rate = len(target) / len(test)

    # Expected number offered among next 30,000
    expected_offers = N_FUTURE * target_rate

    # Incremental profit relative to T=101
    profit_30000 = expected_offers * value_per_targeted

    results.append({
        "T": T,
        "target_rate": target_rate,
        "churn_no_offer": p0,
        "churn_offer": p1,
        "drop_in_churn": churn_drop,
        "value_per_targeted": value_per_targeted,
        "expected_offers_30000": expected_offers,
        "profit_30000": profit_30000
    })

results = pd.DataFrame(results)

# Find optimal threshold
best = results.loc[results["profit_30000"].idxmax()]

print("OPTIMAL POLICY FOR NEXT 30,000")
print("--------------------------------")
print(f"Optimal T: {best['T']:.0f}")
print(f"Offer rule: risk_score >= {best['T']:.0f}")
print(f"Expected offer rate: {best['target_rate']:.1%}")
print(f"Expected offers: {best['expected_offers_30000']:,.0f}")

if best["T"] != 101:
    print(f"Churn without offer: {best['churn_no_offer']:.1%}")
    print(f"Churn with offer: {best['churn_offer']:.1%}")
    print(f"Drop in churn: {best['drop_in_churn']:.1%}")
    print(f"Value per targeted customer: ${best['value_per_targeted']:,.2f}")

print(
    f"Predicted incremental profit vs. T=101: "
    f"${best['profit_30000']:,.2f}"
)

# Show the best alternatives
print("\nTOP 10 THRESHOLDS")
print(
    results[
        [
            "T",
            "target_rate",
            "drop_in_churn",
            "value_per_targeted",
            "expected_offers_30000",
            "profit_30000"
        ]
    ]
    .sort_values("profit_30000", ascending=False)
    .head(10)
    .round(4)
    .to_string(index=False)
)

OPTIMAL POLICY FOR NEXT 30,000
--------------------------------
Optimal T: 35
Offer rule: risk_score >= 35
Expected offer rate: 92.7%
Expected offers: 27,806
Churn without offer: 42.4%
Churn with offer: 30.2%
Drop in churn: 12.1%
Value per targeted customer: $28.27
Predicted incremental profit vs. T=101: $786,085.82

TOP 10 THRESHOLDS
 T  target_rate  drop_in_churn  value_per_targeted  expected_offers_30000  profit_30000
35       0.9269         0.1214             28.2701               27806.25   786085.8194
40       0.8554         0.1230             29.8761               25661.25   766656.9915
34       0.9404         0.1196             26.7415               28211.25   754412.3387
39       0.8709         0.1216             28.7202               26126.25   750351.5172
36       0.9129         0.1199             27.1832               27386.25   744445.6169
38       0.8858         0.1207             27.9630               26572.50   743047.9062
44       0.8000         0.1235             30.6

In [16]:
N_FUTURE = 30000

# ----------------------------------------------------------
# Baseline: T = 101 (nobody gets an offer)
# ----------------------------------------------------------
# Expected churn rate without an offer from randomized controls

p_churn_no_offer = test.loc[
    test["assigned_discount"] == 0,
    "churned_90d"
].mean()

# Under no offer:
# retained customer -> $800
# churned customer  -> $0
#
# Change this if your baseline profit definition differs.

profit_per_customer_T101 = (
    (1 - p_churn_no_offer) * 800
)

profit_T101 = (
    N_FUTURE * profit_per_customer_T101
)


# ----------------------------------------------------------
# Find optimal threshold using the results calculated earlier
# ----------------------------------------------------------

best = results.loc[
    results["profit_30000"].idxmax()
]

optimal_T = best["T"]

# "profit_30000" in our previous calculation is the
# incremental value of the offer policy relative to no offers.

incremental_value_optimal = best["profit_30000"]

profit_optimal = (
    profit_T101 + incremental_value_optimal
)


# ----------------------------------------------------------
# Compare
# ----------------------------------------------------------

comparison = pd.DataFrame({
    "Policy": [
        f"Optimal T = {optimal_T:.0f}",
        "T = 101 (no offers)"
    ],
    "Predicted profit": [
        profit_optimal,
        profit_T101
    ]
})

print(comparison.to_string(index=False))

print("\n----------------------------------")
print(f"Profit at optimal T: ${profit_optimal:,.2f}")
print(f"Profit at T=101:     ${profit_T101:,.2f}")

print(
    f"Incremental profit:  "
    f"${profit_optimal - profit_T101:,.2f}"
)

             Policy  Predicted profit
     Optimal T = 35      1.489435e+07
T = 101 (no offers)      1.410827e+07

----------------------------------
Profit at optimal T: $14,894,353.54
Profit at T=101:     $14,108,267.72
Incremental profit:  $786,085.82


In [13]:
import pandas as pd
import numpy as np

# Create the requested risk bands
bins = [30, 50, 70, 85, 101]
labels = ["30–49", "50–69", "70–84", "85–100"]

d = test[test["risk_score"].between(30, 100)].copy()

d["risk_band"] = pd.cut(
    d["risk_score"],
    bins=bins,
    labels=labels,
    right=False
)

rows = []

for band in labels:
    g = d[d["risk_band"] == band]

    # Churn rates from randomized experiment
    p0 = g.loc[
        g["assigned_discount"] == 0,
        "churned_90d"
    ].mean()

    p1 = g.loc[
        g["assigned_discount"] == 1,
        "churned_90d"
    ].mean()

    # Reduction in churn caused by offer
    churn_drop = p0 - p1

    # Value/profit difference per customer
    value = (
        churn_drop * 800
        - (1 - p1) * 90
        - p1 * 20
    )

    rows.append({
        "Risk band": band,
        "Churn, no offer": p0,
        "Churn, offer": p1,
        "Drop in churn": churn_drop,
        "Value per customer": value
    })

result = pd.DataFrame(rows)

# Format like your table
formatted = result.copy()

formatted["Churn, no offer"] = (
    formatted["Churn, no offer"].map(lambda x: f"{x:.1%}")
)

formatted["Churn, offer"] = (
    formatted["Churn, offer"].map(lambda x: f"{x:.1%}")
)

formatted["Drop in churn"] = (
    formatted["Drop in churn"].map(lambda x: f"{x:.1%}")
)

formatted["Value per customer"] = (
    formatted["Value per customer"].map(lambda x: f"${x:,.2f}")
)

print(formatted.to_string(index=False))

Risk band Churn, no offer Churn, offer Drop in churn Value per customer
    30–49           33.2%        23.7%          9.5%              $2.54
    50–69           38.8%        28.1%         10.7%             $15.43
    70–84           43.2%        30.8%         12.4%             $30.59
   85–100           53.2%        39.9%         13.3%             $44.32


## Submit (round 2)

**[Open the submission form](https://docs.google.com/forms/d/e/1FAIpQLSf-87eDsEHRVat_a4le91sC7bfZFel9fAHNmMtHVuGJzqaHOQ/viewform)** and enter the values printed above, exactly as shown.

- **Numbers only.** Type `150000`, not `$150,000`, `150K` or words.
- **Case:** `1`.  **Round:** `2` (Part II).
- **Submissions are final.** You can't change an answer after you submit, so check the printed values first. If your team submits more than once for the same round, only the **first** submission counts.
- Use the **same team name** in every round.

## Reflection (submit)

1. Your Part 1 analysis may have fit the data very well. Why was it still the wrong tool for this decision?
2. In one sentence: what was the difference between **predicting** the outcome and **deciding** what to do?
3. Name one other business decision where the same trap could appear.